# Machine Learning-Based Cardiometabolic Risk Stratification Using Population Health Data: An Explainable Clinical Decision Support System

## Data Loading and Cleaning

**Student Name:** Vanshika Karnwal | Student ID: 201957572  
**Supervisor:** Dr Baoru Huang  
**Module:** COMP702 Dissertation | University of Liverpool  
**Dataset:** NHANES 2017-March 2020 Pre-Pandemic  
**Source:** https://wwwn.cdc.gov/nchs/nhanes/Default.aspx

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded")

Libraries loaded


## 1. Load Raw NHANES Files

Eleven NHANES 2017-March 2020 Pre-Pandemic component files are loaded covering 
eight core clinical domains and three lifestyle/metabolic expansion files.
All files were downloaded from the CDC NHANES public repository.

In [2]:
BASE = r'C:\Users\VANSHIKA KARNWAL\OneDrive\Desktop\DISSERTATION\DATASETS'

# 8 CORE NHANES FILES 
demo   = pd.read_sas(f'{BASE}\\Demographic dataset.xpt', format='xport')
bpx    = pd.read_sas(f'{BASE}\\Blood_pressure dataset.xpt', format='xport')
bmx    = pd.read_sas(f'{BASE}\\Body_measures dataset.xpt', format='xport')
ghb    = pd.read_sas(f'{BASE}\\Glycohemoglobin (HbA1c) dataset.xpt', format='xport')
glu    = pd.read_sas(f'{BASE}\\Plasma_fasting_glucose.xpt', format='xport')
tchol  = pd.read_sas(f'{BASE}\\Total_cholesterol dataset.xpt', format='xport')
hdl    = pd.read_sas(f'{BASE}\\HDL dataset.xpt', format='xport')
trigly = pd.read_sas(f'{BASE}\\Triglycerides_LDL dataset.xpt', format='xport')

# 3 EXPANDED FILES 
smq    = pd.read_sas(f'{BASE}\\Smoking dataset (Questionnaire).xpt', format='xport')
paq    = pd.read_sas(f'{BASE}\\Physical activity dataset(Questionnaire) .xpt', format='xport')
biopro = pd.read_sas(f'{BASE}\\Biochemistry dataset.xpt', format='xport')

print("All 11 files loaded successfully")

print(f"\nFile shapes:")
print(f"  Demographics:      {demo.shape}")
print(f"  Blood Pressure:    {bpx.shape}")
print(f"  Body Measures:     {bmx.shape}")
print(f"  HbA1c:             {ghb.shape}")
print(f"  Fasting Glucose:   {glu.shape}")
print(f"  Total Cholesterol: {tchol.shape}")
print(f"  HDL Cholesterol:   {hdl.shape}")
print(f"  Triglycerides/LDL: {trigly.shape}")
print(f"  Smoking:           {smq.shape}")
print(f"  Physical Activity: {paq.shape}")
print(f"  Biochemistry:      {biopro.shape}")

All 11 files loaded successfully

File shapes:
  Demographics:      (15560, 29)
  Blood Pressure:    (11656, 12)
  Body Measures:     (14300, 22)
  HbA1c:             (10409, 2)
  Fasting Glucose:   (5090, 4)
  Total Cholesterol: (12198, 3)
  HDL Cholesterol:   (12198, 3)
  Triglycerides/LDL: (5090, 10)
  Smoking:           (11137, 16)
  Physical Activity: (9693, 17)
  Biochemistry:      (10409, 41)


## 2. Variable Selection

Clinically relevant variables are extracted from each file based on 
established cardiometabolic risk factors. SEQN (participant sequence number) 
is retained throughout all files as the unique identifier for merging.

| File | Variables Selected | Clinical Rationale |
|------|-------------------|-------------------|
| DEMO | Age, Gender, Race | Core demographic risk factors |
| BPX | Systolic/Diastolic BP | Averaged across 3 readings |
| BMX | BMI, Waist Circumference | Obesity markers |
| GHB | HbA1c | Glycaemic control marker |
| GLU | Fasting Glucose | Diabetes diagnostic marker |
| TCHOL | Total Cholesterol | Lipid risk marker |
| HDL | HDL Cholesterol | Protective lipid marker |
| TRIGLY | Triglycerides, LDL | Dyslipidaemia markers |
| SMQ | Smoking Status | Major modifiable CVD risk |
| PAQ | Physical Activity, Sedentary Hours | Lifestyle markers |
| BIOPRO | Creatinine, Uric Acid, GGT | Renal and hepatic markers |

In [3]:
# DEMOGRAPHICS
# RIDAGEYR: Age in years | RIAGENDR: Gender (1=Male, 2=Female)
# RIDRETH3: Race/Ethnicity (includes Non-Hispanic Asian category)
demo_clean = demo[['SEQN', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH3']].copy()
demo_clean.columns = ['SEQN', 'Age', 'Gender', 'Race']
demo_clean = demo_clean[demo_clean['Age'] >= 18]  # Adults only

# BLOOD PRESSURE 
# Three readings taken per participant - averaged for reliability
# BPXOSY: Systolic | BPXODI: Diastolic
bpx_clean = bpx[['SEQN', 'BPXOSY1','BPXOSY2','BPXOSY3',
                   'BPXODI1','BPXODI2','BPXODI3']].copy()
bpx_clean['Systolic_BP']  = bpx_clean[['BPXOSY1','BPXOSY2','BPXOSY3']].mean(axis=1)
bpx_clean['Diastolic_BP'] = bpx_clean[['BPXODI1','BPXODI2','BPXODI3']].mean(axis=1)
bpx_clean = bpx_clean[['SEQN', 'Systolic_BP', 'Diastolic_BP']]

# BODY MEASURES
# BMXBMI: Body Mass Index (kg/m²) | BMXWAIST: Waist Circumference (cm)
bmx_clean = bmx[['SEQN', 'BMXBMI', 'BMXWAIST']].copy()
bmx_clean.columns = ['SEQN', 'BMI', 'Waist_Circumference']

# GLYCOHAEMOGLOBIN (HbA1c) 
# LBXGH: HbA1c (%) - gold standard for glycaemic control assessment
ghb_clean = ghb[['SEQN', 'LBXGH']].copy()
ghb_clean.columns = ['SEQN', 'HbA1c']

# FASTING GLUCOSE
# LBXGLU: Plasma fasting glucose (mg/dL)
glu_clean = glu[['SEQN', 'LBXGLU']].copy()
glu_clean.columns = ['SEQN', 'Fasting_Glucose']

# TOTAL CHOLESTEROL
# LBXTC: Total cholesterol (mg/dL)
tchol_clean = tchol[['SEQN', 'LBXTC']].copy()
tchol_clean.columns = ['SEQN', 'Total_Cholesterol']

# HDL CHOLESTEROL
# LBDHDD: Direct HDL cholesterol (mg/dL) - higher values are protective
hdl_clean = hdl[['SEQN', 'LBDHDD']].copy()
hdl_clean.columns = ['SEQN', 'HDL_Cholesterol']

# TRIGLYCERIDES AND LDL
# LBXTR: Triglycerides (mg/dL)
# LBDLDL: LDL cholesterol calculated using Friedewald equation
# LDL = Total Cholesterol - HDL - (Triglycerides / 5)
trigly_clean = trigly[['SEQN', 'LBXTR', 'LBDLDL']].copy()
trigly_clean.columns = ['SEQN', 'Triglycerides', 'LDL_Cholesterol']

# SMOKING STATUS
# SMQ020: Ever smoked 100+ cigarettes (1=Yes, 2=No)
# SMQ040: Current smoking status (1=Every day, 2=Some days, 3=Not at all)
def assign_smoking(row):
    if row['SMQ020'] == 2.0:
        return 'Never'
    elif row['SMQ020'] == 1.0 and row['SMQ040'] == 3.0:
        return 'Former'
    elif row['SMQ020'] == 1.0 and row['SMQ040'] in [1.0, 2.0]:
        return 'Current'
    else:
        return None

smq['Smoking_Status'] = smq.apply(assign_smoking, axis=1)
smq_clean = smq[['SEQN', 'Smoking_Status']].copy()

# PHYSICAL ACTIVITY
# WHO formula: vigorous activity counts double moderate activity
# 9999 = don't know/refused - replaced with NaN
activity_cols = ['PAD615', 'PAD630', 'PAD645', 'PAD660', 'PAD675', 'PAD680']
paq[activity_cols] = paq[activity_cols].replace(9999, np.nan)
paq['Total_PA_Minutes'] = (
    paq['PAD615'].fillna(0) * 2 +   # vigorous work x2
    paq['PAD630'].fillna(0) +         # moderate work
    paq['PAD645'].fillna(0) * 2 +   # vigorous recreational x2
    paq['PAD660'].fillna(0) +         # moderate recreational
    paq['PAD675'].fillna(0)           # walking/cycling
)
paq['Sedentary_Hours'] = paq['PAD680'] / 60
paq['Sedentary_Hours'] = paq['Sedentary_Hours'].clip(upper=16)  # cap at 16 hrs
paq_clean = paq[['SEQN', 'Total_PA_Minutes', 'Sedentary_Hours']].copy()

# BIOCHEMISTRY
# LBXSCR: Serum creatinine (mg/dL) - used to calculate eGFR
# LBXSUA: Uric acid (mg/dL) - linked to gout and metabolic syndrome
# LBXSGTSI: GGT (U/L) - liver enzyme, metabolic dysfunction marker
biopro_clean = biopro[['SEQN', 'LBXSCR', 'LBXSUA', 'LBXSGTSI']].copy()
biopro_clean.columns = ['SEQN', 'Creatinine', 'Uric_Acid', 'GGT']

print("Variable selection complete")

print("\nShapes:")
print("Demographics:     ", demo_clean.shape)
print("Blood Pressure:   ", bpx_clean.shape)
print("Body Measures:    ", bmx_clean.shape)
print("HbA1c:            ", ghb_clean.shape)
print("Fasting Glucose:  ", glu_clean.shape)
print("Total Cholesterol:", tchol_clean.shape)
print("HDL Cholesterol:  ", hdl_clean.shape)
print("Triglycerides/LDL:", trigly_clean.shape)
print("Smoking:          ", smq_clean.shape)
print("Physical Activity:", paq_clean.shape)
print("Biochemistry:     ", biopro_clean.shape)

Variable selection complete

Shapes:
Demographics:      (9693, 4)
Blood Pressure:    (11656, 3)
Body Measures:     (14300, 3)
HbA1c:             (10409, 2)
Fasting Glucose:   (5090, 2)
Total Cholesterol: (12198, 2)
HDL Cholesterol:   (12198, 2)
Triglycerides/LDL: (5090, 3)
Smoking:           (11137, 2)
Physical Activity: (9693, 3)
Biochemistry:      (10409, 4)


## 3. Dataset Merging

All files are merged using SEQN via inner join, retaining only participants 
with data across all clinical domains. Fasting-dependent variables (Glucose, 
Triglycerides/LDL) and biochemistry markers reduce the sample to participants 
who completed the full examination protocol including fasting subsample.

In [4]:
# Merge all 11 files on SEQN using inner join
merged = demo_clean.copy()

for df_temp in [bpx_clean, bmx_clean, ghb_clean, glu_clean,
                tchol_clean, hdl_clean, trigly_clean,
                smq_clean, paq_clean, biopro_clean]:
    merged = merged.merge(df_temp, on='SEQN', how='inner')

print("Merged shape:", merged.shape)
print("Columns:", merged.columns.tolist())

Merged shape: (4438, 20)
Columns: ['SEQN', 'Age', 'Gender', 'Race', 'Systolic_BP', 'Diastolic_BP', 'BMI', 'Waist_Circumference', 'HbA1c', 'Fasting_Glucose', 'Total_Cholesterol', 'HDL_Cholesterol', 'Triglycerides', 'LDL_Cholesterol', 'Smoking_Status', 'Total_PA_Minutes', 'Sedentary_Hours', 'Creatinine', 'Uric_Acid', 'GGT']


## 4. Feature Engineering

### 4.1 eGFR Calculation
eGFR is calculated using the CKD-EPI 2021 equation - the current gold 
standard recommended by NICE NG203 and KDIGO 2021 guidelines (Inker et al., 2021).

### 4.2 Categorical Variable Decoding
Gender and Race are decoded from NHANES numeric codes to readable labels.

### 4.3 Risk Outcome Variable
A three-tier cardiometabolic risk outcome is defined using established 
clinical thresholds:
- **High:** HbA1c ≥6.5% (ADA 2024) OR Fasting Glucose ≥126 mg/dL (ADA 2024) OR Systolic BP ≥140 mmHg (NICE NG133) OR BMI ≥35 kg/m² (WHO)
- **Moderate:** HbA1c 5.7-6.4% OR Fasting Glucose 100-125 mg/dL OR Systolic BP 130-139 mmHg OR BMI 30-34.9 kg/m²
- **Low:** All values within normal ranges

In [5]:
# eGFR CALCULATION (CKD-EPI 2021) 

def calculate_egfr(row):
    """
    CKD-EPI 2021 equation -  NICE NG203 / KDIGO 2021 recommended
    Reference: Inker et al. (2021) New England Journal of Medicine
    """
    creatinine = row['Creatinine']
    age = row['Age']
    gender = row['Gender']  # 1.0 = Male, 2.0 = Female 

    if pd.isna(creatinine) or pd.isna(age) or pd.isna(gender):
        return np.nan

    if gender == 2.0:  # Female
        kappa, alpha, sex_factor = 0.7, -0.241, 1.012
    else:  # Male
        kappa, alpha, sex_factor = 0.9, -0.302, 1.0

    ratio = creatinine / kappa
    if ratio < 1:
        egfr = 142 * (ratio ** alpha) * (0.9938 ** age) * sex_factor
    else:
        egfr = 142 * (ratio ** -1.200) * (0.9938 ** age) * sex_factor

    return round(egfr, 1)

merged['eGFR'] = merged.apply(calculate_egfr, axis=1)
print(f"eGFR calculated - {merged['eGFR'].notna().sum():,} valid values")

# DECODE CATEGORICAL VARIABLES (after eGFR) 
merged['Gender'] = merged['Gender'].map({1.0: 'Male', 2.0: 'Female'})
merged['Race'] = merged['Race'].map({
    1.0: 'Mexican American',
    2.0: 'Other Hispanic',
    3.0: 'Non-Hispanic White',
    4.0: 'Non-Hispanic Black',
    6.0: 'Non-Hispanic Asian',
    7.0: 'Other/Multi-Racial'
})
print("Categorical variables decoded")

# RISK OUTCOME VARIABLE 
def assign_risk(row):
    """
    Three-tier cardiometabolic risk classification
    Based on ADA 2024, NICE NG133, and WHO guidelines
    """
    if (row['HbA1c'] >= 6.5 or row['Fasting_Glucose'] >= 126 or
            row['Systolic_BP'] >= 140 or row['BMI'] >= 35):
        return 'High'
    elif (row['HbA1c'] >= 5.7 or row['Fasting_Glucose'] >= 100 or
          row['Systolic_BP'] >= 130 or row['BMI'] >= 30):
        return 'Moderate'
    else:
        return 'Low'

merged['Risk_category'] = merged.apply(assign_risk, axis=1)
print("Risk categories assigned")

print(f"\nRisk Distribution:")
for cat in ['Low', 'Moderate', 'High']:
    n = (merged['Risk_category'] == cat).sum()
    pct = n / len(merged) * 100
    print(f"  {cat}: {n:,} ({pct:.1f}%)")

eGFR calculated - 4,094 valid values
Categorical variables decoded
Risk categories assigned

Risk Distribution:
  Low: 919 (20.7%)
  Moderate: 1,681 (37.9%)
  High: 1,838 (41.4%)


## 5. Data Cleaning and Final Dataset

Listwise deletion is applied. Participants missing any clinical variable 
are excluded. This preserves data integrity without imputation bias, 
which is particularly important for clinical biomarkers where imputed 
values could misrepresent true physiological status.

In [6]:
# MISSING VALUE ANALYSIS 
print("Missing values before cleaning:")
print(merged.isnull().sum())

# LISTWISE DELETION
# Drop missing values
final = merged.dropna().copy()

print("Final dataset shape:", final.shape)
print("Missing values:", final.isnull().sum().sum())
print("\nGender:", final['Gender'].value_counts().to_dict())
print("\nRace:", final['Race'].value_counts().to_dict())
print("\nRisk Distribution:", final['Risk_category'].value_counts().to_dict())

# Save
save_path = r'C:\Users\VANSHIKA KARNWAL\OneDrive\Desktop\DISSERTATION\DATASETS\nhanes_final_dataset_dissertation.csv'
final.to_csv(save_path, index=False)
print("\nDataset saved")

Missing values before cleaning:
SEQN                     0
Age                      0
Gender                   0
Race                     0
Systolic_BP            455
Diastolic_BP           455
BMI                     90
Waist_Circumference    237
HbA1c                  234
Fasting_Glucose        265
Total_Cholesterol      327
HDL_Cholesterol        327
Triglycerides          338
LDL_Cholesterol        370
Smoking_Status           3
Total_PA_Minutes         0
Sedentary_Hours         34
Creatinine             344
Uric_Acid              346
GGT                    346
eGFR                   344
Risk_category            0
dtype: int64
Final dataset shape: (3528, 22)
Missing values: 0

Gender: {'Female': 1783, 'Male': 1745}

Race: {'Non-Hispanic White': 1220, 'Non-Hispanic Black': 894, 'Mexican American': 442, 'Non-Hispanic Asian': 426, 'Other Hispanic': 353, 'Other/Multi-Racial': 193}

Risk Distribution: {'High': 1470, 'Moderate': 1391, 'Low': 667}

Dataset saved
